In [ ]:
import json
import random
import string
from pathlib import Path

from datasets import load_dataset


def format_choices_with_letters(choices):
    """
    Convert a list like:
        ["left", "right", "above", "below"]
    into:
        A) left
        B) right
        C) above
        D) below
    """
    if len(choices) > 26:
        raise ValueError("Too many choices: supports at most 26 options (A-Z).")

    lines = []
    for i, choice in enumerate(choices):
        letter = string.ascii_uppercase[i]
        lines.append(f"{letter}) {choice}")
    return "\n".join(lines)


def build_prompt(question, choices):
    """
    Build the final prompt:
    <question>

    Answer with the option's letter from the given choices directly.

    A) ...
    B) ...
    ...
    """
    choice_block = format_choices_with_letters(choices)
    return (
        f"{question}\n\n"
        f"Answer with the option's letter from the given choices directly.\n\n"
        f"{choice_block}"
    )


def extract_choices_from_example(ex):
    """
    spatial-mm stores options as separate columns:
      option_1, option_2, option_3, option_4
    Some rows may have missing/empty options, so we filter them out.
    """
    choices = []
    for key in ["option_1", "option_2", "option_3", "option_4"]:
        val = ex.get(key)
        if val is not None:
            val = str(val).strip()
            if val != "":
                choices.append(val)
    return choices


def has_valid_mcq_fields(ex):
    question = ex.get("question")
    image = ex.get("image")
    answer = ex.get("answer")
    choices = extract_choices_from_example(ex)

    return (
        isinstance(question, str)
        and question.strip() != ""
        and image is not None
        and isinstance(answer, str)
        and answer.strip().upper() in {"A", "B", "C", "D"}
        and len(choices) >= 2
    )


def sample_spatial_mm_to_jsonl(
    output_jsonl="spatial_mm_sample_1000.jsonl",
    num_samples=1000,
    seed=42,
    image_output_dir="spatial_mm_sample_images",
    dataset_name="advaitgupta/spatial-mm",
    split="train",
):
    """
    Sample random distinct examples from spatial-mm and save as JSONL.

    Output format per line:
    {
        "id": "...",
        "image_path": "...",
        "prompt": "..."
    }
    """
    random.seed(seed)

    ds = load_dataset(dataset_name, split=split)

    valid_indices = [i for i in range(len(ds)) if has_valid_mcq_fields(ds[i])]

    if num_samples > len(valid_indices):
        raise ValueError(
            f"Requested {num_samples} samples, but only {len(valid_indices)} valid rows "
            f"were found in split '{split}'."
        )

    sampled_indices = random.sample(valid_indices, num_samples)

    image_output_dir = Path(image_output_dir)
    image_output_dir.mkdir(parents=True, exist_ok=True)

    output_path = Path(output_jsonl)

    with output_path.open("w", encoding="utf-8") as f:
        for i, idx in enumerate(sampled_indices, start=1):
            ex = ds[idx]

            # Save image locally
            img = ex["image"]  # PIL Image
            local_image_name = f"img{i}.png"
            local_image_path = image_output_dir / local_image_name
            img.save(local_image_path)

            question = ex["question"]
            choices = extract_choices_from_example(ex)
            prompt = build_prompt(question, choices)

            # Traceable id
            example_id = f"row{idx}__ans{str(ex.get('answer', 'NA')).strip().upper()}"

            record = {
                "id": example_id,
                "image_path": str(local_image_path),
                "prompt": prompt,
            }

            f.write(json.dumps(record, ensure_ascii=False) + "\n")

    print(f"Loaded split '{split}' from {dataset_name}")
    print(f"Found {len(valid_indices)} valid rows")
    print(f"Saved {num_samples} samples to {output_path}")
    print(f"Saved images under {image_output_dir}")


if __name__ == "__main__":
    sample_spatial_mm_to_jsonl()

In [ ]:
import json
import re
import string
from pathlib import Path
from collections import Counter

from datasets import load_dataset


def normalize_text(s: str):
    if s is None:
        return None
    return " ".join(str(s).strip().lower().split())


def extract_option_letter(text: str, num_choices: int = None):
    """
    Try to parse the model output as a letter choice.

    Supports outputs such as:
      A
      a
      A)
      A.
      (A)
      Answer: A
      The answer is B

    Returns uppercase letter if found, else None.

    If num_choices is given, only returns letters within the valid range.
    """
    if text is None:
        return None

    text = text.strip()

    valid_letters = string.ascii_uppercase
    if num_choices is not None:
        valid_letters = valid_letters[:num_choices]

    # strict direct one-letter style
    m = re.fullmatch(r"\s*[\(\[]?([A-Za-z])[\)\]\.\:]?\s*", text)
    if m:
        letter = m.group(1).upper()
        if letter in valid_letters:
            return letter

    # common answer patterns
    patterns = [
        r"\banswer\s*[:\-]?\s*([A-Za-z])\b",
        r"\boption\s*[:\-]?\s*([A-Za-z])\b",
        r"\bchoice\s*[:\-]?\s*([A-Za-z])\b",
        r"\bthe answer is\s*([A-Za-z])\b",
        r"\b([A-Za-z])[\)\.]\s*",
    ]

    for pat in patterns:
        m = re.search(pat, text, flags=re.IGNORECASE)
        if m:
            letter = m.group(1).upper()
            if letter in valid_letters:
                return letter

    # fallback: first standalone single-letter token
    for m in re.finditer(r"\b([A-Za-z])\b", text):
        letter = m.group(1).upper()
        if letter in valid_letters:
            return letter

    return None


def extract_choice_by_text(text: str, choices):
    """
    Fallback parser:
    if the model outputs the actual choice text instead of the letter,
    try to match it against one of the choices.

    Returns the matched choice index, else None.
    """
    if text is None:
        return None

    norm_pred = normalize_text(text)
    if not norm_pred:
        return None

    normalized_choices = [normalize_text(c) for c in choices]

    # strict exact match
    for i, c in enumerate(normalized_choices):
        if norm_pred == c:
            return i

    # substring matching
    for i, c in enumerate(normalized_choices):
        if c and c in norm_pred:
            return i

    return None


def extract_choices_from_example(ex):
    """
    spatial-mm stores options in separate columns:
      option_1, option_2, option_3, option_4
    """
    choices = []
    for key in ["option_1", "option_2", "option_3", "option_4"]:
        val = ex.get(key)
        if val is not None:
            val = str(val).strip()
            if val != "":
                choices.append(val)
    return choices


def has_valid_mcq_fields(ex):
    question = ex.get("question")
    answer = ex.get("answer")
    image = ex.get("image")
    choices = extract_choices_from_example(ex)

    return (
        isinstance(question, str)
        and question.strip() != ""
        and image is not None
        and isinstance(answer, str)
        and answer.strip().upper() in {"A", "B", "C", "D"}
        and len(choices) >= 2
    )


def build_gt_index(dataset_name="advaitgupta/spatial-mm", split="train"):
    """
    Build a mapping from the custom example_id format used in the prep script
    to the ground-truth information.

    Expected example_id format:
      row123__ansB
    """
    ds = load_dataset(dataset_name, split=split)

    gt_index = {}

    for row_idx, ex in enumerate(ds):
        if not has_valid_mcq_fields(ex):
            continue

        choices = extract_choices_from_example(ex)
        gt_letter = str(ex["answer"]).strip().upper()

        if gt_letter not in {"A", "B", "C", "D"}:
            continue

        gt_index_num = string.ascii_uppercase.index(gt_letter)
        gt_answer_text = (
            choices[gt_index_num] if 0 <= gt_index_num < len(choices) else None
        )

        example_id = f"row{row_idx}__ans{gt_letter}"

        gt_index[example_id] = {
            "row_index": row_idx,
            "question": ex.get("question"),
            "choices": choices,
            "gt_answer_index": gt_index_num,
            "gt_answer_letter": gt_letter,
            "gt_answer_text": gt_answer_text,
        }

    return gt_index


def evaluate_result_folder(
    results_dir,
    output_json_path="evaluation_results_spatial_mm.json",
    dataset_name="advaitgupta/spatial-mm",
    split="train",
):
    results_dir = Path(results_dir)
    json_files = sorted(results_dir.glob("*.json"))

    if not json_files:
        raise ValueError(f"No JSON files found in: {results_dir}")

    gt_index = build_gt_index(dataset_name=dataset_name, split=split)

    per_example = {}
    stats = Counter()

    for json_file in json_files:
        try:
            with open(json_file, "r", encoding="utf-8") as f:
                data = json.load(f)
        except Exception as e:
            per_example[json_file.name] = {
                "error": f"Could not read JSON: {e}",
                "correct": None,
            }
            stats["read_error"] += 1
            continue

        example_id = data.get("example", {}).get("example_id")
        raw_answer = data.get("chat_result", {}).get("answer")

        # fallback to full_text if answer missing
        if raw_answer is None:
            raw_answer = data.get("chat_result", {}).get("full_text")

        if example_id is None:
            per_example[json_file.name] = {
                "example_id": None,
                "pred_answer_raw": raw_answer,
                "pred_answer_letter": None,
                "pred_answer_index": None,
                "gt_answer_index": None,
                "gt_answer_letter": None,
                "correct": None,
                "error": "Missing example.example_id",
            }
            stats["missing_example_id"] += 1
            continue

        if example_id not in gt_index:
            per_example[json_file.name] = {
                "example_id": example_id,
                "pred_answer_raw": raw_answer,
                "pred_answer_letter": None,
                "pred_answer_index": None,
                "gt_answer_index": None,
                "gt_answer_letter": None,
                "correct": None,
                "error": "example_id not found in spatial-mm index",
            }
            stats["missing_gt"] += 1
            continue

        gt = gt_index[example_id]
        num_choices = len(gt["choices"])

        pred_letter = extract_option_letter(raw_answer, num_choices=num_choices)
        pred_index = None
        parse_mode = None

        if pred_letter is not None:
            pred_index = string.ascii_uppercase.index(pred_letter)
            parse_mode = "letter"
        else:
            matched_idx = extract_choice_by_text(raw_answer, gt["choices"])
            if matched_idx is not None:
                pred_index = matched_idx
                pred_letter = string.ascii_uppercase[matched_idx]
                parse_mode = "choice_text"

        if pred_index is None:
            per_example[json_file.name] = {
                "example_id": example_id,
                "pred_answer_raw": raw_answer,
                "pred_answer_letter": None,
                "pred_answer_index": None,
                "gt_answer_index": gt["gt_answer_index"],
                "gt_answer_letter": gt["gt_answer_letter"],
                "gt_answer_text": gt["gt_answer_text"],
                "choices": gt["choices"],
                "correct": 0,
                "error": "Could not parse model answer as a valid option letter or choice text",
                "question": gt["question"],
                "row_index": gt["row_index"],
            }
            stats["unparsed_prediction"] += 1
            stats["evaluated"] += 1
            stats["incorrect"] += 1
            continue

        correct = 1 if pred_index == gt["gt_answer_index"] else 0

        per_example[json_file.name] = {
            "example_id": example_id,
            "pred_answer_raw": raw_answer,
            "pred_answer_letter": pred_letter,
            "pred_answer_index": pred_index,
            "pred_answer_text": gt["choices"][pred_index]
            if 0 <= pred_index < len(gt["choices"])
            else None,
            "gt_answer_index": gt["gt_answer_index"],
            "gt_answer_letter": gt["gt_answer_letter"],
            "gt_answer_text": gt["gt_answer_text"],
            "choices": gt["choices"],
            "correct": correct,
            "parse_mode": parse_mode,
            "question": gt["question"],
            "row_index": gt["row_index"],
        }

        stats["evaluated"] += 1
        if correct == 1:
            stats["correct"] += 1
        else:
            stats["incorrect"] += 1

    accuracy = (
        stats["correct"] / stats["evaluated"]
        if stats["evaluated"] > 0
        else 0.0
    )

    output = {
        "summary": {
            "results_dir": str(results_dir),
            "dataset_name": dataset_name,
            "split": split,
            "num_json_files": len(json_files),
            "evaluated": stats["evaluated"],
            "correct": stats["correct"],
            "incorrect": stats["incorrect"],
            "accuracy": accuracy,
            "missing_example_id": stats["missing_example_id"],
            "missing_gt": stats["missing_gt"],
            "unparsed_prediction": stats["unparsed_prediction"],
            "read_error": stats["read_error"],
        },
        "per_example": per_example,
    }

    with open(output_json_path, "w", encoding="utf-8") as f:
        json.dump(output, f, ensure_ascii=False, indent=2)

    print(f"Saved evaluation to: {output_json_path}")
    print(f"Accuracy: {accuracy:.4f} ({stats['correct']}/{stats['evaluated']})")


if __name__ == "__main__":
    evaluate_result_folder(
        results_dir="content_mm_spatial/outputs_json",
        output_json_path="evaluation_results_spatial_mm.json",
        dataset_name="advaitgupta/spatial-mm",
        split="train",
    )